# Análise Exploratória de Dados, UCI Online Retail II

Vou começar entendendo o que tenho em mãos antes de qualquer transformação. Dados de varejo costumam vir bagunçados: transações canceladas misturadas com vendas reais, SKUs mal cadastrados, datas fora de ordem. Preciso mapear tudo isso antes de avançar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

## 1. Carregando os dados

O dataset UCI Online Retail II cobre transações de um varejista online do Reino Unido entre 2009 e 2011. São mais de 500 mil registros. Vou carregar e já verificar a estrutura básica.

In [ ]:
# Carrego os dois anos disponíveis e junto tudo num único DataFrame
df_2009 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_2010 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')

df = pd.concat([df_2009, df_2010], ignore_index=True)

print(f'Linhas: {df.shape[0]:,}')
print(f'Colunas: {df.shape[1]}')
df.head()

## 2. Entendendo cada variável

Antes de qualquer limpeza, preciso entender o que cada coluna representa na prática operacional.

In [ ]:
# Tipos de dados e contagem de nulos, primeiro diagnóstico
print('Tipos de dados:')
print(df.dtypes)
print()
print('Valores nulos por coluna:')
print(df.isnull().sum())
print()
print('Percentual nulo:')
print((df.isnull().sum() / len(df) * 100).round(2))

In [ ]:
# Duplicatas, em datasets de transação isso é comum quando o sistema de origem
# registra o mesmo evento mais de uma vez por falha de integração
duplicatas = df.duplicated().sum()
print(f'Linhas duplicadas: {duplicatas:,} ({duplicatas/len(df)*100:.1f}% do total)')

## 3. Estatísticas descritivas

Quero ver a distribuição de Quantity e Price antes de qualquer limpeza. Valores negativos em Quantity indicam devoluções ou cancelamentos, isso vai aparecer aqui.

In [ ]:
df.describe()

In [ ]:
# Transações com quantity negativo, são cancelamentos, identificados pelo prefixo 'C' no Invoice
cancelamentos = df[df['Quantity'] < 0]
print(f'Transações canceladas: {len(cancelamentos):,}')
print(f'Percentual do total: {len(cancelamentos)/len(df)*100:.1f}%')

# Verifico se cancelamentos batem com o prefixo C no Invoice
print(f"\nCancelamentos com Invoice começando em 'C': {df['Invoice'].astype(str).str.startswith('C').sum():,}")

## 4. Distribuição temporal das vendas

Sazonalidade é o coração de qualquer problema de previsão de demanda. Preciso ver como as vendas se distribuem ao longo do tempo antes de escolher o modelo de forecasting.

In [ ]:
# Extraio features temporais que vou precisar em todas as análises seguintes
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df['Hour'] = df['InvoiceDate'].dt.hour

# Vendas por mês, só transações reais, sem cancelamentos
df_vendas = df[df['Quantity'] > 0].copy()
df_vendas['Revenue'] = df_vendas['Quantity'] * df_vendas['Price']

vendas_mensais = df_vendas.groupby(['Year', 'Month'])['Revenue'].sum().reset_index()
vendas_mensais['Periodo'] = pd.to_datetime(
    vendas_mensais['Year'].astype(str) + '-' + vendas_mensais['Month'].astype(str).str.zfill(2)
)

fig, ax = plt.subplots()
ax.plot(vendas_mensais['Periodo'], vendas_mensais['Revenue'], marker='o', linewidth=2)
ax.set_title('Receita mensal, UCI Online Retail II')
ax.set_xlabel('Período')
ax.set_ylabel('Receita (GBP)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/receita_mensal.png', dpi=150)
plt.show()

print('O pico de outubro/novembro é consistente com o padrão de varejo online, '
      'antecipação de compras de fim de ano.')

## 5. Análise por SKU, identificando os produtos críticos

Para previsão de demanda, não preciso modelar todos os SKUs com a mesma prioridade. Vou aplicar uma análise ABC para separar os produtos que respondem pela maior parte da receita, esses são os que mais causam impacto quando entram em ruptura.

In [ ]:
# Receita por SKU
receita_sku = df_vendas.groupby('StockCode')['Revenue'].sum().sort_values(ascending=False).reset_index()
receita_sku['cumsum_pct'] = receita_sku['Revenue'].cumsum() / receita_sku['Revenue'].sum() * 100

# Classificação ABC
receita_sku['Classe'] = 'C'
receita_sku.loc[receita_sku['cumsum_pct'] <= 80, 'Classe'] = 'A'
receita_sku.loc[(receita_sku['cumsum_pct'] > 80) & (receita_sku['cumsum_pct'] <= 95), 'Classe'] = 'B'

print('Distribuição ABC:')
print(receita_sku['Classe'].value_counts())
print()

# Curva de Pareto
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

ax1.bar(range(len(receita_sku[:100])), receita_sku['Revenue'][:100], color='steelblue', alpha=0.7)
ax2.plot(range(len(receita_sku[:100])), receita_sku['cumsum_pct'][:100], color='crimson', linewidth=2)

ax1.set_xlabel('SKU (top 100)')
ax1.set_ylabel('Receita (GBP)')
ax2.set_ylabel('% Acumulado')
ax2.axhline(80, color='gray', linestyle='--', alpha=0.7, label='80%')
plt.title('Curva de Pareto, Receita por SKU')
plt.tight_layout()
plt.savefig('../reports/figures/pareto_skus.png', dpi=150)
plt.show()

## 6. Análise de correlações

Quero entender a relação entre preço, quantidade e receita. Isso vai me ajudar a decidir quais features incluir nos modelos de forecasting.

In [ ]:
# Correlação entre variáveis numéricas principais
cols_numericas = ['Quantity', 'Price', 'Revenue']
corr_matrix = df_vendas[cols_numericas].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlação entre variáveis numéricas')
plt.tight_layout()
plt.savefig('../reports/figures/correlacao.png', dpi=150)
plt.show()

## 7. Padrão por dia da semana e hora

Isso me diz quando as vendas se concentram. Para modelos com granularidade diária ou horária, esse padrão vai entrar como feature sazonal.

In [ ]:
dias = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']

fig, axes = plt.subplots(1, 2)

vendas_dia = df_vendas.groupby('DayOfWeek')['Revenue'].mean()
axes[0].bar(dias[:len(vendas_dia)], vendas_dia.values, color='steelblue', alpha=0.8)
axes[0].set_title('Receita média por dia da semana')
axes[0].tick_params(axis='x', rotation=45)

vendas_hora = df_vendas.groupby('Hour')['Revenue'].mean()
axes[1].plot(vendas_hora.index, vendas_hora.values, marker='o', color='steelblue')
axes[1].set_title('Receita média por hora do dia')
axes[1].set_xlabel('Hora')

plt.tight_layout()
plt.savefig('../reports/figures/sazonalidade_semanal_horaria.png', dpi=150)
plt.show()

## 8. Conclusões da EDA

Com essa análise exploratória, consigo extrair alguns pontos importantes antes de partir para a modelagem:

- O dataset tem sazonalidade clara com pico no quarto trimestre, isso favorece Prophet e ARIMA com componente sazonal
- Aproximadamente 20% dos SKUs respondem por 80% da receita, a análise ABC vai guiar a priorização dos modelos de forecasting
- Há cancelamentos que precisam ser tratados antes de qualquer modelagem (quantity negativo + prefixo C no Invoice)
- O padrão horário mostra concentração entre 9h e 15h, útil para granularidade intradiária se necessário

O próximo notebook vai fazer a limpeza e a engenharia de features com base nessas observações.